# 🎨 Taller – Pintura Interactiva con Voz y Gestos

**Integrantes:**
- Joan Sebastian Roberto Puerto
- Baruj Vladimir Ramírez Escalante
- Diego Alberto Romero Olmos
- Maicol Sebastian Olarte Ramirez
- Jorge Isaac Alandete Díaz

**Fecha de entrega:** 25 de abril de 2026

---

## Descripción

Este taller crea una obra artística digital controlada por **gestos de la mano** (MediaPipe) y **comandos de voz** (speech_recognition).  
El dedo índice actúa como pincel sobre un lienzo OpenCV. Los comandos de voz cambian el color, borran el lienzo o guardan la obra.  
Un hilo de escucha de voz corre en paralelo para no bloquear el bucle de video en tiempo real.


## 1. Instalación de dependencias

Ejecuta esta celda una sola vez para instalar todas las bibliotecas necesarias.  
En Linux también se requiere `portaudio19-dev` para `pyaudio`:

```bash
sudo apt-get install -y portaudio19-dev python3-pyaudio
```


In [ ]:
import subprocess, sys

def pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *packages])

pip_install(
    "mediapipe",
    "opencv-python",
    "SpeechRecognition",
    "pyaudio",
    "numpy",
)
print("✓ Todas las dependencias instaladas.")


## 2. Importaciones y configuración global


In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import speech_recognition as sr
import threading
import queue
import time
import os
from pathlib import Path

# ── Paleta de colores (BGR) ────────────────────────────────────────────────
COLORES = {
    "rojo":     (0,   0,   255),
    "verde":    (0,   255, 0  ),
    "azul":     (255, 0,   0  ),
    "amarillo": (0,   255, 255),
    "blanco":   (255, 255, 255),
    "negro":    (0,   0,   0  ),
    "magenta":  (255, 0,   255),
    "cian":     (255, 255, 0  ),
}

# ── Tipos de pincel disponibles ────────────────────────────────────────────
BRUSH_CIRCLE  = "circle"   # Círculo relleno (por defecto)
BRUSH_SQUARE  = "square"   # Cuadrado relleno (palma abierta)
BRUSH_LINE    = "line"     # Trazo continuo fino

# ── Estado compartido (accedido desde hilos de voz y video) ───────────────
estado = {
    "color":        COLORES["rojo"],
    "nombre_color": "rojo",
    "grosor":       12,
    "brush":        BRUSH_CIRCLE,
    "limpiar":      False,
    "guardar":      False,
    "feedback":     "",
    "feedback_ts":  0.0,   # timestamp del último mensaje
}

estado_lock = threading.Lock()   # protege escrituras concurrentes
cola_voz    = queue.Queue()      # comandos producidos por el hilo de voz

CARPETA_MEDIA = Path(__file__).resolve().parent.parent / "media" \
    if "__file__" in dir() else Path("../media")
CARPETA_MEDIA.mkdir(parents=True, exist_ok=True)

print("✓ Bibliotecas importadas.")
print(f"✓ Carpeta media: {CARPETA_MEDIA}")


## 3. Sistema de reconocimiento de voz

El hilo de voz escucha el micrófono de forma continua y deposita los comandos reconocidos en `cola_voz`.  
El mapa de comandos admite sinónimos para tolerar errores de transcripción.


In [ ]:
# ── Mapa de palabras clave → acción interna ────────────────────────────────
MAPA_COMANDOS = {
    # Colores
    "rojo":     ("COLOR", "rojo"),
    "red":      ("COLOR", "rojo"),
    "verde":    ("COLOR", "verde"),
    "green":    ("COLOR", "verde"),
    "azul":     ("COLOR", "azul"),
    "blue":     ("COLOR", "azul"),
    "amarillo": ("COLOR", "amarillo"),
    "yellow":   ("COLOR", "amarillo"),
    "blanco":   ("COLOR", "blanco"),
    "white":    ("COLOR", "blanco"),
    "negro":    ("COLOR", "negro"),
    "black":    ("COLOR", "negro"),
    "magenta":  ("COLOR", "magenta"),
    "cian":     ("COLOR", "cian"),
    # Acciones
    "limpiar":  ("ACCION", "limpiar"),
    "borrar":   ("ACCION", "limpiar"),
    "clear":    ("ACCION", "limpiar"),
    "guardar":  ("ACCION", "guardar"),
    "save":     ("ACCION", "guardar"),
    "salir":    ("ACCION", "salir"),
    "exit":     ("ACCION", "salir"),
    # Pinceles
    "cuadrado": ("PINCEL", BRUSH_SQUARE),
    "square":   ("PINCEL", BRUSH_SQUARE),
    "círculo":  ("PINCEL", BRUSH_CIRCLE),
    "circulo":  ("PINCEL", BRUSH_CIRCLE),
    "línea":    ("PINCEL", BRUSH_LINE),
    "linea":    ("PINCEL", BRUSH_LINE),
    "line":     ("PINCEL", BRUSH_LINE),
    # Tamaño
    "grande":   ("TAMANIO", 20),
    "big":      ("TAMANIO", 20),
    "pequeño":  ("TAMANIO", 6),
    "pequeno":  ("TAMANIO", 6),
    "small":    ("TAMANIO", 6),
    "mediano":  ("TAMANIO", 12),
    "medium":   ("TAMANIO", 12),
}


def interpretar_texto(texto: str):
    """Devuelve una tupla (tipo, valor) o None si no hay coincidencia."""
    t = texto.lower().strip()
    for palabra, accion in MAPA_COMANDOS.items():
        if palabra in t:
            return accion
    return None


def aplicar_comando(cmd):
    """Aplica una acción al estado compartido (llamado desde el hilo de video)."""
    if cmd is None:
        return
    tipo, valor = cmd
    with estado_lock:
        if tipo == "COLOR":
            estado["color"]        = COLORES[valor]
            estado["nombre_color"] = valor
            estado["feedback"]     = f"Color: {valor}"
        elif tipo == "ACCION":
            if valor == "limpiar":
                estado["limpiar"]  = True
                estado["feedback"] = "Lienzo limpiado"
            elif valor == "guardar":
                estado["guardar"]  = True
                estado["feedback"] = "Guardando obra..."
        elif tipo == "PINCEL":
            estado["brush"]    = valor
            estado["feedback"] = f"Pincel: {valor}"
        elif tipo == "TAMANIO":
            estado["grosor"]   = valor
            estado["feedback"] = f"Tamaño: {valor}"
        estado["feedback_ts"] = time.time()


# ── Hilo de escucha continua ───────────────────────────────────────────────
_escucha_activa = threading.Event()
_escucha_activa.set()   # inicia activo

def _hilo_voz():
    """Hilo productor: escucha el micrófono y deposita comandos en cola_voz."""
    recognizer = sr.Recognizer()
    recognizer.pause_threshold   = 0.6   # fin de frase más rápido
    recognizer.dynamic_energy_threshold = True

    while _escucha_activa.is_set():
        try:
            with sr.Microphone() as fuente:
                recognizer.adjust_for_ambient_noise(fuente, duration=0.5)
                audio = recognizer.listen(fuente, timeout=5, phrase_time_limit=4)
            texto = recognizer.recognize_google(audio, language="es-ES")
            print(f"[VOZ] Escuché: '{texto}'")
            cmd = interpretar_texto(texto)
            if cmd:
                cola_voz.put(cmd)
        except sr.WaitTimeoutError:
            pass   # silencio prolongado, reintentar
        except sr.UnknownValueError:
            pass   # no se entendió el audio
        except sr.RequestError as e:
            print(f"[VOZ] Error de servicio: {e}")
            time.sleep(2)
        except Exception as e:
            print(f"[VOZ] Error inesperado: {e}")
            time.sleep(1)

print("✓ Sistema de voz definido. El hilo inicia al ejecutar la celda principal.")


## 4. Detección de gestos con MediaPipe Hands

MediaPipe detecta 21 puntos de referencia (*landmarks*) en cada mano.  
Se usan los siguientes gestos:

| Gesto | Detección | Acción |
|---|---|---|
| Dedo índice extendido | Punta del índice (tip) > nudo (pip) en Y | Dibuja con el pincel actual |
| Palma abierta (≥ 4 dedos) | 4 dedos extendidos | Cambia a pincel cuadrado |
| Puño cerrado | 0-1 dedos extendidos | Pausa el dibujo |


In [ ]:
mp_hands   = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_styles  = mp.solutions.drawing_styles

# Índices de landmarks usados
TIP_IDS = [
    mp_hands.HandLandmark.INDEX_FINGER_TIP,
    mp_hands.HandLandmark.MIDDLE_FINGER_TIP,
    mp_hands.HandLandmark.RING_FINGER_TIP,
    mp_hands.HandLandmark.PINKY_TIP,
]
PIP_IDS = [
    mp_hands.HandLandmark.INDEX_FINGER_PIP,
    mp_hands.HandLandmark.MIDDLE_FINGER_PIP,
    mp_hands.HandLandmark.RING_FINGER_PIP,
    mp_hands.HandLandmark.PINKY_PIP,
]


def contar_dedos_extendidos(landmarks, ancho, alto):
    """Devuelve cuántos dedos (sin el pulgar) están extendidos."""
    extendidos = 0
    for tip_id, pip_id in zip(TIP_IDS, PIP_IDS):
        tip_y = landmarks[tip_id].y * alto
        pip_y = landmarks[pip_id].y * alto
        if tip_y < pip_y:   # tip más arriba que pip → extendido
            extendidos += 1
    return extendidos


def obtener_posicion_indice(landmarks, ancho, alto):
    """Devuelve (x, y) en píxeles de la punta del índice."""
    lm = landmarks[mp_hands.HandLandmark.INDEX_FINGER_TIP]
    return int(lm.x * ancho), int(lm.y * alto)


print("✓ Utilidades MediaPipe definidas.")


## 5. Lienzo digital y funciones de dibujo

El lienzo es un array NumPy de tres canales (BGR) que se superpone sobre el fotograma de la cámara con `cv2.addWeighted`.  
Las funciones de dibujo soportan tres tipos de pincel: círculo, cuadrado y línea.


In [ ]:
def crear_lienzo(alto, ancho):
    """Devuelve un lienzo negro (uint8, BGR)."""
    return np.zeros((alto, ancho, 3), dtype=np.uint8)


def dibujar_punto(lienzo, pos, color, grosor, tipo_pincel, pos_anterior=None):
    """
    Dibuja un punto o segmento en el lienzo según el tipo de pincel.

    Args:
        lienzo:       Array numpy del lienzo (modificado in-place).
        pos:          (x, y) posición actual del pincel en píxeles.
        color:        Tupla BGR del color de trazado.
        grosor:       Radio/tamaño del pincel en píxeles.
        tipo_pincel:  BRUSH_CIRCLE | BRUSH_SQUARE | BRUSH_LINE
        pos_anterior: (x, y) posición del fotograma anterior (para líneas continuas).
    """
    x, y = pos
    if tipo_pincel == BRUSH_CIRCLE:
        cv2.circle(lienzo, (x, y), grosor, color, -1, cv2.LINE_AA)
    elif tipo_pincel == BRUSH_SQUARE:
        half = grosor
        cv2.rectangle(lienzo,
                      (x - half, y - half),
                      (x + half, y + half),
                      color, -1)
    elif tipo_pincel == BRUSH_LINE:
        if pos_anterior:
            cv2.line(lienzo, pos_anterior, (x, y), color, max(2, grosor // 3), cv2.LINE_AA)
        else:
            cv2.circle(lienzo, (x, y), max(2, grosor // 3), color, -1, cv2.LINE_AA)


def componer_frame(frame, lienzo, alpha=0.75):
    """Superpone el lienzo sobre el frame de la cámara."""
    # Solo mezcla donde el lienzo tiene píxeles pintados (no negros)
    mascara = cv2.cvtColor(lienzo, cv2.COLOR_BGR2GRAY)
    _, mascara = cv2.threshold(mascara, 1, 255, cv2.THRESH_BINARY)
    resultado = frame.copy()
    resultado[mascara > 0] = cv2.addWeighted(
        frame, 1 - alpha, lienzo, alpha, 0
    )[mascara > 0]
    return resultado


def guardar_obra(lienzo, frame, carpeta: Path):
    """Guarda el lienzo y la composición final como archivos .png."""
    ts = time.strftime("%Y%m%d_%H%M%S")
    ruta_lienzo = carpeta / f"obra_{ts}_lienzo.png"
    ruta_comp   = carpeta / f"obra_{ts}_composicion.png"
    comp = componer_frame(frame, lienzo)
    cv2.imwrite(str(ruta_lienzo), lienzo)
    cv2.imwrite(str(ruta_comp),   comp)
    print(f"[SAVE] Guardado: {ruta_lienzo.name} | {ruta_comp.name}")
    return str(ruta_comp)


print("✓ Lienzo y funciones de dibujo definidos.")


## 6. HUD – Retroalimentación visual en pantalla

El HUD (*Heads-Up Display*) muestra en tiempo real:
- Barra de color activo (parte superior izquierda).
- Tipo y tamaño de pincel.
- Último comando de voz reconocido (desaparece tras 2 segundos).
- Mini-paleta de colores disponibles.


In [ ]:
FONT       = cv2.FONT_HERSHEY_SIMPLEX
FONT_SCALE = 0.65
FONT_THICK = 2
DURACION_FEEDBACK = 2.0   # segundos que permanece el mensaje de voz


def dibujar_hud(frame, est):
    """
    Superpone el HUD de estado sobre el frame compuesto.

    Args:
        frame: Frame BGR (modificado in-place).
        est:   Copia del estado actual (dict).
    """
    h, w = frame.shape[:2]

    # ── Panel superior semitransparente ───────────────────────────────────
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (w, 60), (30, 30, 30), -1)
    cv2.addWeighted(overlay, 0.5, frame, 0.5, 0, frame)

    # ── Muestra de color activo ────────────────────────────────────────────
    cv2.rectangle(frame, (10, 10), (50, 50), est["color"], -1)
    cv2.rectangle(frame, (10, 10), (50, 50), (200, 200, 200), 1)

    # ── Información textual ────────────────────────────────────────────────
    info = (f"Color: {est['nombre_color']}  |  "
            f"Pincel: {est['brush']}  |  "
            f"Grosor: {est['grosor']}")
    cv2.putText(frame, info, (60, 38), FONT, FONT_SCALE,
                (220, 220, 220), FONT_THICK, cv2.LINE_AA)

    # ── Feedback de último comando de voz ─────────────────────────────────
    elapsed = time.time() - est["feedback_ts"]
    if est["feedback"] and elapsed < DURACION_FEEDBACK:
        alpha_fade = max(0.0, 1.0 - elapsed / DURACION_FEEDBACK)
        overlay2 = frame.copy()
        cv2.putText(overlay2, f"[VOZ] {est['feedback']}",
                    (10, h - 15), FONT, 0.8,
                    (0, 255, 100), FONT_THICK + 1, cv2.LINE_AA)
        cv2.addWeighted(overlay2, alpha_fade, frame, 1 - alpha_fade, 0, frame)

    # ── Mini paleta en esquina inferior derecha ───────────────────────────
    tam_muestra = 24
    margen      = 6
    nombres_col = list(COLORES.keys())
    for i, nombre in enumerate(nombres_col):
        x1 = w - (len(nombres_col) - i) * (tam_muestra + margen)
        y1 = h - tam_muestra - 10
        x2 = x1 + tam_muestra
        y2 = y1 + tam_muestra
        cv2.rectangle(frame, (x1, y1), (x2, y2), COLORES[nombre], -1)
        borde = (0, 255, 0) if nombre == est["nombre_color"] else (80, 80, 80)
        cv2.rectangle(frame, (x1, y1), (x2, y2), borde, 2)

    # ── Instrucciones rápidas (esquina inferior izquierda) ─────────────────
    instrucciones = [
        "Indice extendido: dibujar",
        "Palma abierta: pincel cuadrado",
        "Puno cerrado: pausa",
        "Voz: rojo|verde|azul|limpiar|guardar",
    ]
    for j, linea in enumerate(instrucciones):
        cv2.putText(frame, linea,
                    (10, h - 120 + j * 22),
                    FONT, 0.45, (180, 180, 180), 1, cv2.LINE_AA)


print("✓ HUD definido.")


## 7. Bucle principal – integración completa en tiempo real

El bucle principal:
1. Captura cada fotograma de la webcam.
2. Detecta landmarks de la mano con MediaPipe.
3. Procesa gestos (dedos extendidos → tipo de pincel / dibujo activo).
4. Consume comandos de voz desde `cola_voz` y los aplica al estado.
5. Dibuja en el lienzo y compone el resultado.
6. Muestra el HUD y presenta la ventana con `cv2.imshow`.

> **Para salir:** presiona `q`, `Esc` o di "salir".


In [ ]:
def ejecutar_pintura_interactiva(camara_id: int = 0):
    """
    Lanza la aplicación de pintura interactiva.

    Args:
        camara_id: Índice de la cámara (0 = predeterminada).
    """
    global _escucha_activa

    # ── Iniciar hilo de voz ────────────────────────────────────────────────
    _escucha_activa.set()
    hilo_voz = threading.Thread(target=_hilo_voz, daemon=True)
    hilo_voz.start()
    print("[MAIN] Hilo de voz iniciado.")

    # ── Abrir webcam ───────────────────────────────────────────────────────
    cap = cv2.VideoCapture(camara_id)
    if not cap.isOpened():
        print("[ERROR] No se pudo abrir la cámara. Verifica el índice.")
        _escucha_activa.clear()
        return

    ancho = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    alto  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    lienzo = crear_lienzo(alto, ancho)

    print(f"[MAIN] Cámara abierta: {ancho}×{alto}")
    print("[MAIN] Presiona 'q' o 'Esc' para salir.")

    pos_anterior = None   # posición del fotograma anterior para trazos continuos
    corriendo    = True
    ultimo_frame = None   # conserva el último frame para guardar la obra

    hands = mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=1,
        min_detection_confidence=0.7,
        min_tracking_confidence=0.6,
    )

    try:
        while corriendo:
            ret, frame = cap.read()
            if not ret:
                print("[WARN] No se recibió fotograma.")
                break

            # Espejo horizontal para experiencia más natural
            frame = cv2.flip(frame, 1)
            ultimo_frame = frame.copy()

            # ── Procesar comandos de voz pendientes ───────────────────────
            while not cola_voz.empty():
                cmd = cola_voz.get_nowait()
                aplicar_comando(cmd)
                # Comando de salida por voz
                tipo, valor = cmd
                if tipo == "ACCION" and valor == "salir":
                    corriendo = False

            # ── Snapshot del estado (evitar race conditions en el render) ──
            with estado_lock:
                est = dict(estado)
                if est["limpiar"]:
                    lienzo = crear_lienzo(alto, ancho)
                    estado["limpiar"] = False
                if est["guardar"]:
                    guardar_obra(lienzo, frame, CARPETA_MEDIA)
                    estado["guardar"] = False

            # ── Detección de manos con MediaPipe ──────────────────────────
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            resultado = hands.process(rgb)

            dibujando      = False
            pos_actual     = None

            if resultado.multi_hand_landmarks:
                landmarks_mano = resultado.multi_hand_landmarks[0]
                lm_list        = landmarks_mano.landmark

                # Dibujar esqueleto de la mano sobre el frame
                mp_drawing.draw_landmarks(
                    frame,
                    landmarks_mano,
                    mp_hands.HAND_CONNECTIONS,
                    mp_styles.get_default_hand_landmarks_style(),
                    mp_styles.get_default_hand_connections_style(),
                )

                dedos = contar_dedos_extendidos(lm_list, ancho, alto)
                pos_actual = obtener_posicion_indice(lm_list, ancho, alto)

                # ── Lógica de gestos ──────────────────────────────────────
                if dedos >= 4:
                    # Palma abierta → pincel cuadrado (bonus)
                    with estado_lock:
                        estado["brush"]    = BRUSH_SQUARE
                        estado["feedback"] = "Gesto: pincel cuadrado"
                        estado["feedback_ts"] = time.time()
                    dibujando = True
                elif dedos == 1:
                    # Solo índice extendido → dibujo normal
                    dibujando = True
                else:
                    # Puño u otros gestos → pausa
                    pos_anterior = None

                # ── Dibujar en el lienzo ──────────────────────────────────
                if dibujando and pos_actual:
                    dibujar_punto(
                        lienzo,
                        pos_actual,
                        est["color"],
                        est["grosor"],
                        est["brush"],
                        pos_anterior if dedos == 1 else None,
                    )
                    pos_anterior = pos_actual
            else:
                pos_anterior = None   # mano fuera de cuadro, reiniciar trazo

            # ── Composición final + HUD ───────────────────────────────────
            frame_comp = componer_frame(frame, lienzo)
            with estado_lock:
                est_hud = dict(estado)
            dibujar_hud(frame_comp, est_hud)

            cv2.imshow("Pintura Interactiva | Voz + Gestos", frame_comp)

            # ── Teclas de salida y acciones rápidas ───────────────────────
            tecla = cv2.waitKey(1) & 0xFF
            if tecla in (ord("q"), 27):   # q o Esc
                corriendo = False
            elif tecla == ord("c"):       # tecla C → limpiar
                lienzo = crear_lienzo(alto, ancho)
            elif tecla == ord("s"):       # tecla S → guardar
                if ultimo_frame is not None:
                    guardar_obra(lienzo, ultimo_frame, CARPETA_MEDIA)

    finally:
        hands.close()
        cap.release()
        cv2.destroyAllWindows()
        _escucha_activa.clear()
        hilo_voz.join(timeout=3)
        print("[MAIN] Aplicación cerrada.")


# ── Punto de entrada ──────────────────────────────────────────────────────
# Descomenta la siguiente línea para lanzar la aplicación desde el notebook:
# ejecutar_pintura_interactiva(camara_id=0)
print("✓ Función principal definida.")
print("Ejecuta: ejecutar_pintura_interactiva() para iniciar la aplicación.")


## 8. (Bonus) Prueba del sistema de voz sin cámara

Esta celda demuestra el reconocimiento de voz de forma aislada, sin necesitar la webcam activa. Captura un único enunciado y muestra el comando interpretado.


In [ ]:
def prueba_voz_aislada():
    """
    Captura un único enunciado del micrófono y muestra el comando interpretado.
    Útil para verificar que speech_recognition funciona correctamente.
    """
    recognizer = sr.Recognizer()
    print("Habla ahora... (esperando hasta 6 segundos)")
    try:
        with sr.Microphone() as fuente:
            recognizer.adjust_for_ambient_noise(fuente, duration=0.5)
            audio = recognizer.listen(fuente, timeout=6, phrase_time_limit=4)
        texto = recognizer.recognize_google(audio, language="es-ES")
        print(f"Reconocido: '{texto}'")
        cmd = interpretar_texto(texto)
        if cmd:
            tipo, valor = cmd
            print(f"→ Comando interpretado: tipo='{tipo}', valor='{valor}'")
        else:
            print("→ No se encontró ningún comando en el enunciado.")
    except sr.WaitTimeoutError:
        print("→ No se detectó audio en el tiempo límite.")
    except sr.UnknownValueError:
        print("→ No se pudo interpretar el audio.")
    except sr.RequestError as e:
        print(f"→ Error del servicio de reconocimiento: {e}")


# Descomenta para probar:
# prueba_voz_aislada()

# ── Prueba del intérprete de texto con frases de ejemplo ─────────────────
print("=== Prueba del intérprete de comandos ===")
frases_prueba = [
    "quiero el color rojo por favor",
    "limpiar el lienzo ahora",
    "guardar obra",
    "azul brillante",
    "pincel cuadrado",
    "tamaño grande",
    "salir de la aplicación",
    "ninguna coincidencia aquí",
]

for frase in frases_prueba:
    cmd = interpretar_texto(frase)
    print(f"  '{frase}' → {cmd}")


## 9. (Bonus) Prueba de detección de gestos con imagen estática

Esta celda demuestra la detección de landmarks de MediaPipe sobre una imagen de prueba cargada desde disco, sin necesitar la webcam en tiempo real.


In [ ]:
import matplotlib.pyplot as plt

def demo_lienzo_sintetico():
    """
    Genera y muestra un lienzo de demostración sin webcam:
    varios trazos de diferentes colores y tipos de pincel.
    """
    alto, ancho = 480, 640
    lienzo_demo = crear_lienzo(alto, ancho)

    # Trazar un arco en rojo con pincel círculo
    for i in range(60):
        x = 80 + i * 8
        y = int(200 - 80 * np.sin(np.radians(i * 3)))
        dibujar_punto(lienzo_demo, (x, y), COLORES["rojo"], 10, BRUSH_CIRCLE,
                      (x - 8, int(200 - 80 * np.sin(np.radians((i - 1) * 3)))) if i > 0 else None)

    # Trazar cuadrados en azul con pincel cuadrado
    for i in range(8):
        cx = 150 + i * 50
        cy = 320
        dibujar_punto(lienzo_demo, (cx, cy), COLORES["azul"], 15, BRUSH_SQUARE)

    # Trazar una línea diagonal en verde con pincel línea
    for i in range(40):
        x = 60 + i * 10
        y = 380 + i * 2
        prev = (x - 10, 378 + (i - 1) * 2) if i > 0 else None
        dibujar_punto(lienzo_demo, (x, y), COLORES["verde"], 8, BRUSH_LINE, prev)

    # Trazar espiral en amarillo
    for angulo in range(0, 720, 5):
        r   = angulo / 15
        cx  = int(520 + r * np.cos(np.radians(angulo)))
        cy  = int(240 + r * np.sin(np.radians(angulo)))
        ang_prev = angulo - 5
        r_prev   = ang_prev / 15
        prev = (int(520 + r_prev * np.cos(np.radians(ang_prev))),
                int(240 + r_prev * np.sin(np.radians(ang_prev)))) if angulo > 0 else None
        if 0 < cx < ancho and 0 < cy < alto:
            dibujar_punto(lienzo_demo, (cx, cy), COLORES["amarillo"], 5, BRUSH_LINE, prev)

    # Mostrar resultado inline
    lienzo_rgb = cv2.cvtColor(lienzo_demo, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(10, 6))
    plt.imshow(lienzo_rgb)
    plt.axis("off")
    plt.title("Lienzo de demostración – distintos pinceles y colores")
    plt.tight_layout()
    plt.show()

    # Guardar el lienzo de demo en media/
    ruta = CARPETA_MEDIA / "demo_lienzo_sintetico.png"
    cv2.imwrite(str(ruta), lienzo_demo)
    print(f"✓ Lienzo de demostración guardado en: {ruta}")


demo_lienzo_sintetico()


## 10. Guardar obra y resumen

La función `guardar_obra()` guarda dos archivos en `media/`:
- `obra_<timestamp>_lienzo.png` — solo las pinceladas sobre fondo negro.
- `obra_<timestamp>_composicion.png` — pinceladas superpuestas sobre el último frame de la cámara.

También puedes presionar `s` en cualquier momento dentro de la ventana interactiva para guardar sin usar la voz.

---

### Cómo ejecutar la aplicación completa

```python
# Ejecuta la celda de la sección 7, luego:
ejecutar_pintura_interactiva(camara_id=0)
```

**Comandos de voz disponibles:**

| Comando | Acción |
|---|---|
| "rojo", "verde", "azul", "amarillo", "blanco", "negro", "magenta", "cian" | Cambia el color del pincel |
| "limpiar" / "borrar" | Borra el lienzo |
| "guardar" | Guarda la obra como PNG en `media/` |
| "cuadrado" / "circulo" / "linea" | Cambia el tipo de pincel |
| "grande" / "mediano" / "pequeño" | Cambia el tamaño del pincel |
| "salir" | Cierra la aplicación |

**Gestos disponibles:**

| Gesto | Acción |
|---|---|
| Dedo índice extendido (1 dedo) | Dibuja con el pincel activo |
| Palma abierta (4+ dedos) | Cambia a pincel cuadrado y dibuja |
| Puño cerrado (0 dedos) | Pausa el dibujo |
